# Mechanical Material Card (PMDCo / IAO): from data entry to RDF

This notebook shows how to assemble a mechanical material card and convert
it into a standardised, machine-readable RDF graph.

**You only need to edit one file:** `docs/example.input.json`. Everything
else is automatic.

A mechanical material card in this schema captures:

- a reference to the **material** it describes (by IRI)
- **elastic constants**: density, Young's modulus, Poisson's ratio
- **mechanical properties** from a tensile test: yield strength, tensile strength,
  elongation (using TTO terms)
- a **constitutive model** with fitted parameter values and a link to the
  calibration step that produced them

Unlike process schemas (manufacturing, characterisation, simulation), a material
card is a **data artefact** (`iao:IAO_0000100`), not a process. It is the
*output* of a calibration step and the *input* to an FEM simulation.


## What the notebook does

```
example.input.json
  |  material IRI, elastic constants, properties, and constitutive model
  |
  v
Transform
  |  converts your plain JSON into a structured OO-LD document
  |
  v
RDF graph
  |  machine-readable, ontology-aligned triples
  |
  v
SHACL validation  ->  confirms the graph is structurally correct
SPARQL inspect    ->  shows the key properties and parameter values
```


## Environment setup

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install jupyterlab
jupyter lab
```

In [ ]:
%pip install -q jsonata-python rdflib pyshacl pyyaml

In [ ]:
import sys, json, pathlib, yaml, pyshacl, rdflib
from jsonata.jsonata import Jsonata
from importlib.metadata import version

print("Python        :", sys.executable)
print("jsonata-python", version("jsonata-python"))
print("rdflib        ", version("rdflib"))
print("pyshacl       ", version("pyshacl"))

HERE   = pathlib.Path().resolve()   # docs/
SCHEMA = HERE.parent                # material-card/mechanical/PMDCo/

## Step 1: Describe your material card

Edit `docs/example.input.json` with your data, then run this cell to load it.

| Field | Required | Description |
|---|---|---|
| `card_name` | yes | A name for this material card |
| `material_iri` | yes | IRI of the material this card describes |
| `description` | no | Free-text explanation |
| `density` | no | Density value and unit (e.g. `{"value": 7.93, "unit": "g/cm³"}`) |
| `youngs_modulus` | no | Young's modulus value and unit (e.g. `{"value": 193.0, "unit": "GPa"}`) |
| `poissons_ratio` | no | Poisson's ratio value (dimensionless) |
| `mechanical_properties` | no | List of TTO property records (see table below) |
| `constitutive_model` | no | Fitted flow curve model with parameter values |
| `card_id` | no | Custom IRI slug; auto-derived from `card_name` if omitted |

**Supported mechanical property types:**

| `type` value | Ontology term | Unit |
|---|---|---|
| `YieldStrength` | TTO | MPa |
| `TensileStrength` | TTO | MPa |
| `PercentageElongationAfterFracture` | TTO | % |
| `PercentageReductionOfArea` | TTO | % |
| `ProofStrength` | TTO | MPa |

In [ ]:
simplified = json.loads((HERE / "example.input.json").read_text())

print(json.dumps(simplified, indent=2))

## Step 2: Convert to OO-LD

This step transforms your plain JSON into a structured
[OO-LD](https://github.com/OO-LD/oold-python) document.
The transform maps short property names (like `YieldStrength`) to their
full TTO ontology IRIs, and simplified unit strings (like `MPa`) to
QUDT unit IRIs.

You can also run the transform from the command line:
```
python -m jsonata -e ../simplified/transform.jsonata -i example.input.json
```

In [ ]:
expr     = (SCHEMA / "simplified" / "transform.jsonata").read_text()
oold_doc = Jsonata(expr).evaluate(simplified)

print(json.dumps(oold_doc, indent=2))

## Step 3: Convert to RDF

The OO-LD document is parsed into an RDF graph using the ontology context from
`specs/schema.oold.yaml`.

The material card node is typed as `iao:IAO_0000100` (DataSet), which is the
standard Information Artifact Ontology class for data artefacts.

In [ ]:
context = yaml.safe_load((SCHEMA / "specs" / "schema.oold.yaml").read_text())["@context"]

g = rdflib.Dataset()
g.parse(data=json.dumps({"@context": context, **oold_doc}), format="json-ld")

print(f"Graph contains {len(g)} triples.\n")
print(g.serialize(format="turtle"))

In [ ]:
# Optional: save to file
out_ttl = HERE / "output_material_card.ttl"
out_ttl.write_text(g.serialize(format="turtle"))
print(f"Written to {out_ttl}")

## Step 4: Validate against the SHACL shape

SHACL (Shapes Constraint Language) defines rules that a valid RDF graph must
satisfy. The shape file `specs/shape.ttl` checks that:

- The material card has exactly one label.
- The `material_iri` reference is an IRI (not a plain string).
- Every scalar property node has a numeric value.
- Every mechanical property node has a numeric value and a unit.

In [ ]:
shapes_graph = rdflib.Graph().parse(str(SCHEMA / "specs" / "shape.ttl"))

conforms, results_graph, _ = pyshacl.validate(
    g,
    shacl_graph = shapes_graph,
)

print(f"Conforms: {conforms}")
if not conforms:
    SH = rdflib.Namespace("http://www.w3.org/ns/shacl#")
    for result in results_graph.subjects(rdflib.RDF.type, SH.ValidationResult):
        msg  = results_graph.value(result, SH.resultMessage)
        path = results_graph.value(result, SH.resultPath)
        loc  = str(path).rsplit("#", 1)[-1].rsplit("/", 1)[-1] if path else None
        print(f"\n  x {msg}" + (f"\n    property: {loc}" if loc else ""))

## Step 5: Inspect the graph

The cells below use SPARQL to confirm that the material card was stored
correctly, including elastic constants, mechanical properties, and
constitutive model parameters.

You do not need to understand SPARQL to use this notebook. Just run the cells
and check that the output matches what you entered.

In [ ]:
# Flatten the Dataset into a plain Graph for SPARQL queries
flat = rdflib.Graph()
for s, p, o, _ in g.quads():
    flat.add((s, p, o))

IAO     = rdflib.Namespace("http://purl.obolibrary.org/obo/IAO_")
DCTERMS = rdflib.Namespace("http://purl.org/dc/terms/")

# Print card label and material reference
card_iri  = next(flat.subjects(rdflib.RDF.type, IAO["0000100"]))
label     = flat.value(card_iri, rdflib.RDFS.label)
mat_ref   = flat.value(card_iri, DCTERMS.references)
print(f"Material card : {card_iri}")
print(f"Label         : {label}")
print(f"Material IRI  : {mat_ref}")

In [ ]:
SPARQL_ELASTIC = """
PREFIX iao:   <http://purl.obolibrary.org/obo/IAO_>
PREFIX pmd:   <https://w3id.org/pmd/co/>
PREFIX qudt:  <http://qudt.org/schema/qudt/>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?property ?value ?unit
WHERE {
  VALUES ?prop { pmd:PMD_0000052  pmd:PMD_0000053  pmd:PMD_0000054 }
  ?card a iao:0000100 ; ?prop ?node .
  ?node qudt:value ?value .
  OPTIONAL { ?node qudt:hasUnit ?unit . }
  BIND(REPLACE(STR(?prop), ".*[/#]", "") AS ?property)
}
ORDER BY ?property
"""

rows = list(flat.query(SPARQL_ELASTIC))
if rows:
    print("Elastic constants:")
    print(f"  {'Property':<22}  {'Value':>10}  Unit")
    print("  " + "-" * 45)
    for r in rows:
        val  = f"{float(r.value):>10.4g}"
        unit = str(r.unit).rsplit("/", 1)[-1] if r.unit else ""
        print(f"  {str(r.property):<22}  {val}  {unit}")
else:
    print("No elastic constants recorded.")

In [ ]:
SPARQL_MECH = """
PREFIX iao:   <http://purl.obolibrary.org/obo/IAO_>
PREFIX pmd:   <https://w3id.org/pmd/co/>
PREFIX qudt:  <http://qudt.org/schema/qudt/>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>
PREFIX tto:   <https://w3id.org/pmd/tto/>

SELECT ?type ?value ?unit
WHERE {
  ?card a iao:0000100 ; pmd:PMD_0000055 ?prop .
  ?prop a ?type ; qudt:value ?value .
  OPTIONAL { ?prop qudt:hasUnit ?unit . }
  FILTER(STRSTARTS(STR(?type), STR(tto:)))
  BIND(REPLACE(STR(?type), STR(tto:), "") AS ?type)
}
ORDER BY ?type
"""

rows = list(flat.query(SPARQL_MECH))
if rows:
    print("Mechanical properties (TTO):")
    print(f"  {'Property type':<40}  {'Value':>10}  Unit")
    print("  " + "-" * 60)
    for r in rows:
        val  = f"{float(r.value):>10.2f}"
        unit = str(r.unit).rsplit("/", 1)[-1] if r.unit else ""
        print(f"  {str(r.type):<40}  {val}  {unit}")
else:
    print("No mechanical properties recorded.")

In [ ]:
SPARQL_MODEL = """
PREFIX iao:   <http://purl.obolibrary.org/obo/IAO_>
PREFIX pmd:   <https://w3id.org/pmd/co/>
PREFIX qudt:  <http://qudt.org/schema/qudt/>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos:  <http://www.w3.org/2004/02/skos/core#>

SELECT ?model_type ?param_name ?param_value ?param_unit
WHERE {
  ?card a iao:0000100 ; pmd:PMD_0000056 ?model .
  ?model skos:notation ?model_type .
  OPTIONAL {
    ?model pmd:PMD_0000057 ?param .
    ?param rdfs:label ?param_name ; qudt:value ?param_value .
    OPTIONAL { ?param qudt:hasUnit ?param_unit . }
  }
}
ORDER BY ?param_name
"""

rows = list(flat.query(SPARQL_MODEL))
if rows:
    model_type = str(rows[0].model_type)
    print(f"Constitutive model: {model_type}")
    print(f"  {'Parameter':<16}  {'Value':>12}  Unit")
    print("  " + "-" * 40)
    for r in rows:
        if r.param_name is None:
            continue
        val  = f"{float(r.param_value):>12.4f}"
        unit = str(r.param_unit).rsplit("/", 1)[-1] if r.param_unit else ""
        print(f"  {str(r.param_name):<16}  {val}  {unit}")
else:
    print("No constitutive model recorded.")

## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.input.json` with the material reference, constants, and model parameters |
| 2 | The data is converted to an OO-LD document (ontology-mapped JSON) |
| 3 | The OO-LD is parsed into an RDF graph |
| 4 | The graph is validated against the SHACL shape |
| 5 | Elastic constants, mechanical properties, and model parameters are extracted to confirm correctness |

To create a card for a different material, edit `docs/example.input.json` and re-run all cells.


## Further reading

- [Schema format reference](../../../docs/schema-format.md): how to write your own schema
- [Schema patterns](../../../docs/schema-patterns.md): inheritance and composition patterns
- `simulation/model-calibration/PMDCo/` is the step that produces this card
- `simulation/step/PMDCo/` is the FEM step that consumes this card
- See `workflow/PMDCo/docs/workflow_demo.ipynb` for a cross-domain demo showing the full production-to-FEM pipeline